# PaySim SQL Exploration

This notebook is the first analysis artifact for `paysim-deep-dive`. The goal is to answer the first five questions a senior analyst should ask after receiving a new payments dataset:

1. What transaction types exist, and where does fraud concentrate?
2. How large are transactions overall and by type?
3. Which originators transact most often?
4. Which destinations receive the most inflow?
5. When does fraud spike across PaySim's simulated time steps?

The notebook uses DuckDB so the 6.3M-row CSV can be queried directly without loading the whole file into pandas first.


## Setup

Expected raw data path: `data/PS_20174392719_1491204439457_log.csv`.

If the file is missing, run from the repo root:

```bash
pip install -r requirements.txt
kaggle datasets download -d ealaxi/paysim1 --unzip -p data/
```


In [10]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

candidate_paths = [
    Path("data/PS_20174392719_1491204439457_log.csv"),
    Path("../data/PS_20174392719_1491204439457_log.csv"),
    Path("data/paysim.csv"),
    Path("../data/paysim.csv"),
]
DATA_PATH = next((path for path in candidate_paths if path.exists()), candidate_paths[0])
DATA_PATH


PosixPath('../data/PS_20174392719_1491204439457_log.csv')

In [11]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"PaySim CSV not found at {DATA_PATH}. "
        "Download it with: kaggle datasets download -d ealaxi/paysim1 --unzip -p data/"
    )

con = duckdb.connect()
con.read_csv(str(DATA_PATH)).create_view("paysim", replace=True)


┌───────┬──────────┬───────────┬─────────────┬───────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────┬────────────────┐
│ step  │   type   │  amount   │  nameOrig   │ oldbalanceOrg │ newbalanceOrig │  nameDest   │ oldbalanceDest │ newbalanceDest │ isFraud │ isFlaggedFraud │
│ int64 │ varchar  │  double   │   varchar   │    double     │     double     │   varchar   │     double     │     double     │  int64  │     int64      │
├───────┼──────────┼───────────┼─────────────┼───────────────┼────────────────┼─────────────┼────────────────┼────────────────┼─────────┼────────────────┤
│     1 │ PAYMENT  │   9839.64 │ C1231006815 │      170136.0 │      160296.36 │ M1979787155 │            0.0 │            0.0 │       0 │              0 │
│     1 │ PAYMENT  │   1864.28 │ C1666544295 │       21249.0 │       19384.72 │ M2044282225 │            0.0 │            0.0 │       0 │              0 │
│     1 │ TRANSFER │     181.0 │ C1305486145 │         181.0 │        

## Dataset Sanity Check

Before analysis, confirm row count, time range, transaction types, and fraud prevalence. This protects the rest of the notebook from silently analyzing the wrong file.


In [12]:
sanity_query = """
SELECT
    COUNT(*) AS rows,
    MIN(step) AS min_step,
    MAX(step) AS max_step,
    COUNT(DISTINCT type) AS transaction_types,
    SUM(isFraud) AS fraud_rows,
    AVG(isFraud) AS fraud_rate
FROM paysim;
"""

sanity = con.sql(sanity_query).df()
sanity


,rows,min_step,max_step,transaction_types,fraud_rows,fraud_rate
0,6362620,1,743,5,8213.0000,0.0013


## Query 1 — Transaction Mix And Fraud Rate By Type

Question: which transaction types dominate volume, and where does fraud actually appear?


In [13]:
query_1 = """
SELECT
    type,
    COUNT(*) AS transactions,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS transaction_share_pct,
    SUM(isFraud) AS fraud_transactions,
    ROUND(100.0 * AVG(isFraud), 4) AS fraud_rate_pct,
    SUM(isFlaggedFraud) AS flagged_transactions
FROM paysim
GROUP BY type
ORDER BY transactions DESC;
"""

q1_type_fraud = con.sql(query_1).df()
q1_type_fraud


,type,transactions,transaction_share_pct,fraud_transactions,fraud_rate_pct,flagged_transactions
0,CASH_OUT,2237500,35.1700,4116.0000,0.1840,0.0000
1,PAYMENT,2151495,33.8100,0.0000,0.0000,0.0000
2,CASH_IN,1399284,21.9900,0.0000,0.0000,0.0000
3,TRANSFER,532909,8.3800,4097.0000,0.7688,16.0000
4,DEBIT,41432,0.6500,0.0000,0.0000,0.0000


**Finding stub:** Fraud should be concentrated in `TRANSFER` and `CASH_OUT`; if other types show fraud, investigate schema or import issues.


## Query 2 — Amount Distribution By Type

Question: do transaction amounts differ enough by type to matter for downstream analysis?


In [14]:
query_2 = """
SELECT
    type,
    COUNT(*) AS transactions,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(MEDIAN(amount), 2) AS median_amount,
    ROUND(QUANTILE_CONT(amount, 0.90), 2) AS p90_amount,
    ROUND(QUANTILE_CONT(amount, 0.99), 2) AS p99_amount,
    ROUND(MAX(amount), 2) AS max_amount
FROM paysim
GROUP BY type
ORDER BY median_amount DESC;
"""

q2_amount_by_type = con.sql(query_2).df()
q2_amount_by_type


,type,transactions,avg_amount,median_amount,p90_amount,p99_amount,max_amount
0,TRANSFER,532909,910647.0100,486308.3900,1782296.6400,10000000.0000,92445516.6400
1,CASH_OUT,2237500,176273.9600,147072.1900,355191.9800,579654.0700,10000000.0000
2,CASH_IN,1399284,168920.2400,143427.7100,343358.1400,550870.8600,1915267.9000
3,PAYMENT,2151495,13057.6000,9482.1900,28810.7300,59500.1100,238637.9800
4,DEBIT,41432,5483.6700,3048.9900,9500.1100,50817.9800,569077.5100


**Finding stub:** Compare median vs p99 to describe skew. This is more useful than reporting only the mean.


## Query 3 — Most Active Originators

Question: are originator IDs mostly one-off accounts, or do repeated originators exist?


In [15]:
query_3 = """
SELECT
    nameOrig,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT type) AS distinct_transaction_types,
    ROUND(SUM(amount), 2) AS total_sent,
    SUM(isFraud) AS fraud_transactions,
    MIN(step) AS first_step,
    MAX(step) AS last_step
FROM paysim
GROUP BY nameOrig
ORDER BY transaction_count DESC, total_sent DESC
LIMIT 20;
"""

q3_top_originators = con.sql(query_3).df()
q3_top_originators


,nameOrig,transaction_count,distinct_transaction_types,total_sent,fraud_transactions,first_step,last_step
0,C545315117,3,3,2485461.6400,0.0000,18,356
1,C724452879,3,1,838815.3000,0.0000,184,544
2,C1902386530,3,2,763712.7900,0.0000,132,400
3,C1976208114,3,3,511130.3500,0.0000,21,330
4,C1530544995,3,2,490535.0900,0.0000,130,228
5,C1784010646,3,3,436700.2500,0.0000,22,309
6,C2098525306,3,1,378102.2200,0.0000,9,354
7,C1832548028,3,3,313665.8300,0.0000,208,416
8,C1999539787,3,2,290555.0100,0.0000,13,139
9,C1677795071,3,2,244735.3800,0.0000,136,212


**Finding stub:** PaySim originators are expected to be near-unique. Repeated originators may still reveal simulator behavior worth noting.


## Query 4 — Top Destinations By Inflow

Question: which destination accounts receive the most value, and are they customers (`C...`) or merchants (`M...`)?


In [16]:
query_4 = """
SELECT
    nameDest,
    CASE
        WHEN STARTS_WITH(nameDest, 'M') THEN 'merchant'
        WHEN STARTS_WITH(nameDest, 'C') THEN 'customer'
        ELSE 'unknown'
    END AS dest_kind,
    COUNT(*) AS incoming_transactions,
    ROUND(SUM(amount), 2) AS total_inflow,
    ROUND(AVG(amount), 2) AS avg_inflow,
    SUM(isFraud) AS fraud_transactions,
    MIN(step) AS first_step,
    MAX(step) AS last_step
FROM paysim
GROUP BY nameDest, dest_kind
ORDER BY total_inflow DESC
LIMIT 20;
"""

q4_top_destinations = con.sql(query_4).df()
q4_top_destinations


,nameDest,dest_kind,incoming_transactions,total_inflow,avg_inflow,fraud_transactions,first_step,last_step
0,C439737079,customer,18,357440831.4400,19857823.9700,0.0000,163,404
1,C707403537,customer,17,299374418.4200,17610259.9100,0.0000,250,642
2,C167875008,customer,28,274736432.8000,9812015.4600,0.0000,39,370
3,C20253152,customer,20,270116188.6900,13505809.4300,0.0000,209,395
4,C172409641,customer,57,255310174.2500,4479125.8600,0.0000,9,402
5,C268913927,customer,35,253484588.1000,7242416.8000,0.0000,36,369
6,C936857833,customer,46,227780012.0200,4951739.3900,0.0000,14,379
7,C65111466,customer,22,227443845.8500,10338356.6300,1.0000,187,533
8,C744189981,customer,26,225173861.7300,8660533.1400,0.0000,48,398
9,C1406193485,customer,46,224778961.8300,4886499.1700,0.0000,7,659


**Finding stub:** Use this to discuss receiver concentration and why merchant/customer prefixes matter for interpretation.


## Query 5 — Fraud Over Simulated Time

Question: does fraud appear evenly across PaySim steps, or are there spikes worth investigating?


In [17]:
query_5 = """
WITH by_step AS (
    SELECT
        step,
        COUNT(*) AS transactions,
        SUM(isFraud) AS fraud_transactions,
        AVG(isFraud) AS fraud_rate
    FROM paysim
    GROUP BY step
)
SELECT
    step,
    transactions,
    fraud_transactions,
    ROUND(100.0 * fraud_rate, 4) AS fraud_rate_pct
FROM by_step
WHERE fraud_transactions > 0
ORDER BY fraud_transactions DESC, fraud_rate DESC
LIMIT 25;
"""

q5_fraud_by_step = con.sql(query_5).df()
q5_fraud_by_step


,step,transactions,fraud_transactions,fraud_rate_pct
0,212,34047,40.0000,0.1175
1,523,30,30.0000,100.0000
2,501,28,28.0000,100.0000
3,387,28,28.0000,100.0000
4,425,28,28.0000,100.0000
5,730,28,28.0000,100.0000
6,249,23209,28.0000,0.1206
7,398,26656,26.0000,0.0975
8,160,27765,26.0000,0.0936
9,66,24,24.0000,100.0000


**Finding stub:** Separate absolute fraud spikes from high-rate low-volume steps. Both can be true, but they imply different operational responses.


## README Findings Draft

After running the notebook, copy 3-5 bullets into `README.md` under a `## Initial findings` section.

Candidate bullets:

- Fraud is not spread across all transaction types; it concentrates in the transaction types that can drain value.
- Transaction amounts are heavily skewed, so medians and tail percentiles are more informative than averages alone.
- Originator behavior is sparse; destination behavior has more repeated entities and is better for concentration analysis.
- Time-step spikes need both count and rate views to avoid overreacting to tiny denominators.
- `isFlaggedFraud` is a very narrow rule baseline, not a full fraud-detection strategy.


## Next Steps

- Add 3-5 findings to `README.md`.
- Add one small chart in a Python EDA notebook only after the SQL notebook is committed.
- Keep modeling out of scope until the EDA story is readable.
